In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
data = pd.read_csv("FinalBalancedDataset.csv")

: 

In [ ]:
data.info()

In [ ]:
data.head(5)

In [ ]:
data = data.drop("Unnamed: 0", axis=1)

In [ ]:
data.head(5)

In [ ]:
data['Toxicity'].value_counts()

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('omw-1.4')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
from nltk import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords as nltk_stopwords
from nltk.corpus import wordnet

## Lemmatizer
Leaves<br>
Leafs<br>Leaf

## Text pre-processing

In [ ]:
wordnet_lemmatizer = WordNetLemmatizer()

In [ ]:
import re

In [ ]:
def prepare_text(text):
    def get_wordnet_pos(treebank_tag):
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN
    text = re.sub(r'[^a-zA-Z\']', ' ', text)
    text = text.split()
    text = ' '.join(text)
    text = word_tokenize(text)
    text = pos_tag(text)
    lemma = []
    for i in text: lemma.append(wordnet_lemmatizer.lemmatize(i[0], pos = get_wordnet_pos(i[1])))
    lemma = ' '.join(lemma)
    return lemma

In [ ]:
data['clean_tweets'] = data['tweet'].apply(lambda x: prepare_text(x))

In [ ]:
data.head(5)

## Tf-Idf for Features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

In [ ]:
corpus = data['clean_tweets'].values.astype('U')

In [ ]:
stopwords = list(nltk_stopwords.words('english'))

In [ ]:
count_tf_idf = TfidfVectorizer(stop_words = stopwords)
tf_idf = count_tf_idf.fit_transform(corpus)

In [ ]:
import pickle

In [ ]:
pickle.dump(count_tf_idf, open("tf_idf.pkt", "wb"))

In [ ]:
tf_idf_train, tf_idf_test, target_train, target_test = train_test_split(
    tf_idf, data['Toxicity'], test_size = 0.2, random_state= 42, shuffle=True
)

## Create a Binary Classification Model

In [ ]:
model_bayes = MultinomialNB()

In [ ]:
model_bayes = model_bayes.fit(tf_idf_train, target_train)

In [ ]:
y_pred_proba = model_bayes.predict_proba(tf_idf_test)[::, 1]

In [ ]:
y_pred_proba

In [ ]:
fpr, tpr, _ = roc_curve(target_test, y_pred_proba)

In [ ]:
final_roc_auc = roc_auc_score(target_test, y_pred_proba)

In [ ]:
print(f'ROC AUC Score: {final_roc_auc}')

# Plot ROC Curve

In [ ]:

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {final_roc_auc:.4f})')
plt.plot([0,1], [0,1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Predict on a New Sample Text

In [ ]:
test_text = "I hate you moron"
print(f"\nPredicting toxicity for the sample text: '{test_text}'")
processed_test_text = prepare_text(test_text)
test_tfidf = count_tf_idf.transform([processed_test_text])
pred_proba = model_bayes.predict_proba(test_tfidf)
pred_label = model_bayes.predict(test_tfidf)
print(f"Prediction Probabilities: {pred_proba}")
print(f"Predicted Label: {pred_label[0]}")  # 0 for non-toxic, 1 for toxic

## Save the model

In [38]:
print("\nSaving the trained model to 'toxicity_model.pkt'...")
with open("toxicity_model.pkt", "wb") as f:
    pickle.dump(model_bayes, f)
print("Model saved successfully.")